# Attention Residual Adapter Analysis

This notebook is organized into two parts.

## Part A: RTE ablation analysis
We only ran the full lookback ablation on `RTE` due to limited compute. This part uses one RTE ablation checkpoint to inspect how different layers use earlier layers.

## Part B: task comparison analysis
We then use the three main AttnRes models (`boolq`, `gsm8k`, `rte`) to compare how attention residual usage differs across tasks.


In [1]:
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Dataset

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = (repo_root / ".." / "..").resolve()

notebook_dir = repo_root / "src" / "analyze"

def resolve_notebook_path(relative_path: str) -> Path:
    return (notebook_dir / relative_path).resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.AttnResAdapter import load_qwen3_attnres_model
from src.analyze.attnres_analyzer import AttnResAnalyzer, AttnResHook
from src.analyze.attnres_visualizer import AttnResVisualizer
from src.utils import build_prompt, load_json, load_model, load_tokenizer

print(f"Repo root: {repo_root}")
print(f"Notebook dir: {notebook_dir}")
print("Imports successful!")


Repo root: /root/project/AttnRes_PEFT
Notebook dir: /root/project/AttnRes_PEFT/src/analyze
Imports successful!


## Step 1: Configure Models and Datasets

We keep one configuration for the RTE ablation analysis and one configuration for the three main task models.


In [5]:
base_model_path = "../../model"
batch_size = 8
max_samples_per_task = 100  # set to None for full analysis

# Part A: use the large-lookback RTE ablation model for detailed layer analysis
rte_ablation_config = {
    "dataset_name": "RTE Ablation",
    "data_path": "../../data/rte/validation.json",
    "adapter_path": "../../checkpoints/ablation/attnres_rte_lookback_28",
    "max_length": 256,
    "lookback": 28,
}

# Part B: use the three main task-specific models for cross-task comparison
main_task_configs = {
    "boolq": {
        "dataset_name": "BoolQ",
        "data_path": "../../data/boolq/validation.json",
        "adapter_path": "../../checkpoints/attnres_boolq",
        "max_length": 512,
        "lookback": 8,
    },
    "gsm8k": {
        "dataset_name": "GSM8K",
        "data_path": "../../data/gsm8k/test.json",
        "adapter_path": "../../checkpoints/attnres_gsm8k",
        "max_length": 1024,
        "lookback": 8,
    },
    "rte": {
        "dataset_name": "RTE",
        "data_path": "../../data/rte/validation.json",
        "adapter_path": "../../checkpoints/attnres_rte",
        "max_length": 256,
        "lookback": 8,
    },
}

device = "cuda" if torch.cuda.is_available() else "cpu"
resolved_base_model_path = str(resolve_notebook_path(base_model_path))
tokenizer = load_tokenizer(resolved_base_model_path)

print(f"Device: {device}")
print(f"Base model path: {base_model_path} -> {resolved_base_model_path}")
print(f"Batch size: {batch_size}")
print(f"Max samples per task: {max_samples_per_task}")
print("\nPart A: RTE ablation model")
print(rte_ablation_config)
print("\nPart B: main task models")
for task_name, cfg in main_task_configs.items():
    print(f"- {task_name}: {cfg['data_path']} | {cfg['adapter_path']} | lookback={cfg['lookback']}")


Device: cuda
Base model path: ../../model -> /root/project/AttnRes_PEFT/model
Batch size: 8
Max samples per task: 100

Part A: RTE ablation model
{'dataset_name': 'RTE Ablation', 'data_path': '../../data/rte/validation.json', 'adapter_path': '../../checkpoints/ablation/attnres_rte_lookback_28', 'max_length': 256, 'lookback': 28}

Part B: main task models
- boolq: ../../data/boolq/validation.json | ../../checkpoints/attnres_boolq | lookback=8
- gsm8k: ../../data/gsm8k/test.json | ../../checkpoints/attnres_gsm8k | lookback=8
- rte: ../../data/rte/validation.json | ../../checkpoints/attnres_rte | lookback=8


In [6]:
def load_attnres_checkpoint(adapter_path, device):
    adapter_dir = resolve_notebook_path(adapter_path)
    safetensors_path = adapter_dir / "model.safetensors"
    pytorch_path = adapter_dir / "pytorch_model.bin"

    if safetensors_path.exists():
        from safetensors.torch import load_file
        return load_file(str(safetensors_path), device=device)

    if pytorch_path.exists():
        return torch.load(str(pytorch_path), map_location=device)

    raise FileNotFoundError(f"Cannot find adapter checkpoint in {adapter_dir}")


def load_task_model(adapter_path, lookback=8, device="cuda"):
    base_model = load_model(resolved_base_model_path, device=device)
    model = load_qwen3_attnres_model(base_model, lookback=lookback, gate_init=0.0)
    state_dict = load_attnres_checkpoint(adapter_path, device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model

print("Model loading helpers ready.")


Model loading helpers ready.


## Step 2: Prepare Datasets

We build one dataloader for the RTE ablation analysis and three dataloaders for the task comparison analysis.


In [7]:
rte_examples = load_json(str(resolve_notebook_path(rte_ablation_config["data_path"])))
if max_samples_per_task is not None:
    rte_examples = rte_examples[:max_samples_per_task]
print(f"RTE ablation set: {len(rte_examples)} examples")

for task_name, cfg in main_task_configs.items():
    examples = load_json(str(resolve_notebook_path(cfg["data_path"])))
    if max_samples_per_task is not None:
        examples = examples[:max_samples_per_task]
    print(f"{task_name}: {len(examples)} examples")


RTE ablation set: 100 examples
boolq: 100 examples
gsm8k: 100 examples
rte: 100 examples


In [8]:
class PromptDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        prompt = build_prompt(example)
        tokens = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
        }

rte_dataset = PromptDataset(rte_examples, tokenizer, max_length=rte_ablation_config["max_length"])
rte_loader = DataLoader(rte_dataset, batch_size=batch_size, shuffle=False)

main_datasets = {}
main_dataloaders = {}
for task_name, cfg in main_task_configs.items():
    examples = load_json(str(resolve_notebook_path(cfg["data_path"])))
    if max_samples_per_task is not None:
        examples = examples[:max_samples_per_task]
    main_datasets[task_name] = examples
    main_dataloaders[task_name] = DataLoader(
        PromptDataset(examples, tokenizer, max_length=cfg["max_length"]),
        batch_size=batch_size,
        shuffle=False,
    )

print("Dataloaders ready.")


Dataloaders ready.


## Step 3: Run the RTE Ablation Analysis

This part uses the large-lookback RTE ablation checkpoint to inspect layer usage in more detail.


In [9]:
def analyze_task(task_name, cfg, dataloader, lookback=None, device="cuda"):
    model = load_task_model(cfg["adapter_path"], lookback=lookback, device=device)
    num_layers = len(model.model.layers)

    analyzer = AttnResAnalyzer(num_layers=num_layers, lookback=lookback)
    analyzer.set_dataset_name(task_name, cfg["dataset_name"])

    print(f"\nAnalyzing {cfg['dataset_name']} ...")
    with torch.no_grad(), AttnResHook(model, analyzer, dataset_id=task_name):
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )
            if (batch_idx + 1) % 10 == 0:
                print(f"  processed {min((batch_idx + 1) * batch_size, len(dataloader.dataset))} samples")

    analyzer.dataset_sample_counts[task_name] = len(dataloader.dataset)
    print(f"✓ {cfg['dataset_name']} analysis complete")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return analyzer

rte_ablation_analyzer = analyze_task(
    "rte_ablation",
    rte_ablation_config,
    rte_loader,
    lookback=rte_ablation_config["lookback"],
    device=device,
)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing RTE Ablation ...
  processed 80 samples
✓ RTE Ablation analysis complete


## Step 4: Compute RTE Ablation Statistics


In [10]:
rte_num_layers = rte_ablation_analyzer.num_layers
rte_ablation_stats = rte_ablation_analyzer.compute_aggregated_stats()
rte_ablation_analyzer.print_summary()



ATTENTION RESIDUAL ADAPTER ANALYSIS SUMMARY

Dataset Overview:
--------------------------------------------------------------------------------
  RTE Ablation: 100 samples

Per-Layer Depth Attention (which earlier layers are attended to):
--------------------------------------------------------------------------------
  Layer 0:
    Most attended sources: [('Embedding', 1.0)]
    Attention distribution: ['1.000']...
  Layer 1:
    Most attended sources: [('Embedding', 0.89599609375), ('Layer 0', 0.10400390625)]
    Attention distribution: ['0.896', '0.104']...
  Layer 2:
    Most attended sources: [('Embedding', 0.845703125), ('Layer 1', 0.0894775390625), ('Layer 0', 0.06488037109375)]
    Attention distribution: ['0.846', '0.065', '0.089']...
  Layer 3:
    Most attended sources: [('Embedding', 0.853515625), ('Layer 2', 0.1005859375), ('Layer 0', 0.0264892578125)]
    Attention distribution: ['0.854', '0.026', '0.019', '0.101']...
  Layer 4:
    Most attended sources: [('Embedding', 

In [11]:
print("RTE ablation statistics ready.")


RTE ablation statistics ready.


## Step 5: Inspect Which Earlier Layers Are Most Used in the RTE Ablation Model


## Important: Lookback Window Constraint

For the RTE ablation model, the valid source layers depend on the chosen lookback window. This section checks whether the attended sources stay inside that valid range.


In [12]:
# Verify lookback constraint for the RTE ablation model (ignore embedding source -1)
print("=" * 80)
print("VALIDATION: Checking Lookback Constraint (lookback={})\n".format(rte_ablation_config["lookback"]))

for check_layer_idx in range(rte_num_layers):
    top_layers = rte_ablation_analyzer.get_most_attended_layers(check_layer_idx, top_k=5)
    valid_start = max(0, check_layer_idx - rte_ablation_config["lookback"])
    valid_end = check_layer_idx - 1

    top_ids_raw = [src_idx for src_idx, _ in top_layers]
    top_ids_layers = [src_idx for src_idx in top_ids_raw if src_idx != -1]

    print(f"Layer {check_layer_idx}: Valid window = [{valid_start}, {valid_end}]")
    print(f"Top attended sources (raw): {top_ids_raw}")
    print(f"Top attended layers (exclude embedding): {top_ids_layers}")

    if valid_end < valid_start:
        all_valid = len(top_ids_layers) == 0
    else:
        all_valid = all(valid_start <= src_idx <= valid_end for src_idx in top_ids_layers)

    print("VALID" if all_valid else "INVALID")
    print()


VALIDATION: Checking Lookback Constraint (lookback=28)

Layer 0: Valid window = [0, -1]
Top attended sources (raw): [-1]
Top attended layers (exclude embedding): []
VALID

Layer 1: Valid window = [0, 0]
Top attended sources (raw): [-1, 0]
Top attended layers (exclude embedding): [0]
VALID

Layer 2: Valid window = [0, 1]
Top attended sources (raw): [-1, 1, 0]
Top attended layers (exclude embedding): [1, 0]
VALID

Layer 3: Valid window = [0, 2]
Top attended sources (raw): [-1, 2, 0, 1]
Top attended layers (exclude embedding): [2, 0, 1]
VALID

Layer 4: Valid window = [0, 3]
Top attended sources (raw): [-1, 3, 2, 0, 1]
Top attended layers (exclude embedding): [3, 2, 0, 1]
VALID

Layer 5: Valid window = [0, 4]
Top attended sources (raw): [-1, 4, 3, 0, 2]
Top attended layers (exclude embedding): [4, 3, 0, 2]
VALID

Layer 6: Valid window = [0, 5]
Top attended sources (raw): [-1, 3, 2, 4, 0]
Top attended layers (exclude embedding): [3, 2, 4, 0]
VALID

Layer 7: Valid window = [0, 6]
Top attende

In [13]:
# Verify the lookback constraint on aggregated RTE ablation stats
print("\n" + "=" * 80)
print("VERIFICATION: Lookback Constraint Applied to RTE Ablation Statistics")
print("=" * 80)

layer_stats = rte_ablation_stats['layer_depth_stats']
test_layers = [0, 5, 8, 9, 14, 16, 18, 20, 23]

for layer_idx in test_layers:
    if layer_idx < rte_num_layers and layer_idx in layer_stats:
        layer_stat = layer_stats[layer_idx]
        valid_start = max(0, layer_idx - rte_ablation_config["lookback"])
        valid_end = layer_idx - 1

        if layer_stat.get('aggregated'):
            ranking = layer_stat['aggregated'].get('layer_ranking', [])[:5]
            print(f"\nLayer {layer_idx}: Valid window = [{valid_start:2d}, {valid_end:2d}]")

            if ranking:
                attended = [f"{'Emb' if src == -1 else f'L{src}'}({w:.3f})" for src, w in ranking]
                print(f"Top 5 attended: {', '.join(attended)}")

                ranking_layers = [(src, w) for src, w in ranking if src != -1]
                if valid_end < valid_start:
                    all_valid = len(ranking_layers) == 0
                else:
                    all_valid = all(valid_start <= src <= valid_end for src, _ in ranking_layers)

                print("VALID" if all_valid else "INVALID")
            else:
                print("No ranking data")



VERIFICATION: Lookback Constraint Applied to RTE Ablation Statistics

Layer 0: Valid window = [ 0, -1]
Top 5 attended: Emb(1.000)
VALID

Layer 5: Valid window = [ 0,  4]
Top 5 attended: Emb(0.854), L4(0.068), L3(0.026), L0(0.024), L2(0.023)
VALID

Layer 8: Valid window = [ 0,  7]
Top 5 attended: Emb(0.552), L7(0.114), L4(0.084), L5(0.075), L3(0.074)
VALID

Layer 9: Valid window = [ 0,  8]
Top 5 attended: Emb(0.217), L8(0.116), L4(0.106), L5(0.106), L0(0.105)
VALID

Layer 14: Valid window = [ 0, 13]
Top 5 attended: L1(0.230), L2(0.165), L0(0.153), L3(0.090), L11(0.063)
VALID

Layer 16: Valid window = [ 0, 15]
Top 5 attended: L0(0.278), L14(0.158), L1(0.147), L2(0.070), L12(0.049)
VALID

Layer 18: Valid window = [ 0, 17]
Top 5 attended: L0(0.189), L1(0.134), Emb(0.090), L2(0.053), L16(0.051)
VALID

Layer 20: Valid window = [ 0, 19]
Top 5 attended: L0(0.439), L1(0.162), L2(0.074), L3(0.053), Emb(0.044)
VALID

Layer 23: Valid window = [ 0, 22]
Top 5 attended: Emb(0.269), L3(0.154), L0(0.1

## Step 6: RTE Ablation Layer Usage Summary


In [14]:
print("\n" + "=" * 80)
print("MOST ATTENDED SOURCES PER ADAPTER LAYER (RTE Ablation)")
print("=" * 80)

for layer_idx in range(rte_num_layers):
    top_layers = rte_ablation_analyzer.get_most_attended_layers(layer_idx, top_k=5)
    print(f"\nAdapter Layer {layer_idx} attends to:")
    for source_idx, weight in top_layers:
        source_name = "Embedding" if source_idx == -1 else f"Layer {source_idx}"
        print(f"  {source_name}: {weight:.4f}")



MOST ATTENDED SOURCES PER ADAPTER LAYER (RTE Ablation)

Adapter Layer 0 attends to:
  Embedding: 1.0000

Adapter Layer 1 attends to:
  Embedding: 0.8960
  Layer 0: 0.1040

Adapter Layer 2 attends to:
  Embedding: 0.8457
  Layer 1: 0.0895
  Layer 0: 0.0649

Adapter Layer 3 attends to:
  Embedding: 0.8535
  Layer 2: 0.1006
  Layer 0: 0.0265
  Layer 1: 0.0192

Adapter Layer 4 attends to:
  Embedding: 0.7021
  Layer 3: 0.1464
  Layer 2: 0.1075
  Layer 0: 0.0237
  Layer 1: 0.0205

Adapter Layer 5 attends to:
  Embedding: 0.8535
  Layer 4: 0.0679
  Layer 3: 0.0259
  Layer 0: 0.0236
  Layer 2: 0.0230

Adapter Layer 6 attends to:
  Embedding: 0.3376
  Layer 3: 0.1638
  Layer 2: 0.1614
  Layer 4: 0.1340
  Layer 0: 0.1074

Adapter Layer 7 attends to:
  Embedding: 0.3821
  Layer 6: 0.1855
  Layer 0: 0.1421
  Layer 5: 0.1140
  Layer 4: 0.0723

Adapter Layer 8 attends to:
  Embedding: 0.5522
  Layer 7: 0.1141
  Layer 4: 0.0845
  Layer 5: 0.0748
  Layer 3: 0.0739

Adapter Layer 9 attends to:
  Embe

In [15]:
# Focused verification for mid/late layers in the RTE ablation model
print("\n" + "=" * 80)
print("FOCUSED CHECK: RTE ablation lookback mapping for layers 14-18")
print("=" * 80)

for layer_idx in [0, 14, 15, 16, 17, 18]:
    top_sources = rte_ablation_analyzer.get_most_attended_layers(layer_idx, top_k=8)
    state_start = max(0, layer_idx + 1 - rte_ablation_config["lookback"])
    state_end = layer_idx
    expected_sources = [(-1 if s == 0 else s - 1) for s in range(state_start, state_end + 1)]

    pretty_expected = ["Embedding" if s == -1 else f"Layer {s}" for s in expected_sources]
    pretty_top = [f"{'Embedding' if s == -1 else f'Layer {s}'}({w:.3f})" for s, w in top_sources]

    print(f"\nAdapter Layer {layer_idx}")
    print(f"  Expected candidate sources: {pretty_expected}")
    print(f"  Top attended sources:      {pretty_top}")

    top_ids = [s for s, _ in top_sources if s != -1]
    expected_layers_only = [s for s in expected_sources if s != -1]
    all_valid = all(s in expected_layers_only for s in top_ids)
    print(f"  Constraint check: {'PASS' if all_valid else 'FAIL'}")



FOCUSED CHECK: RTE ablation lookback mapping for layers 14-18

Adapter Layer 0
  Expected candidate sources: ['Embedding']
  Top attended sources:      ['Embedding(1.000)']
  Constraint check: PASS

Adapter Layer 14
  Expected candidate sources: ['Embedding', 'Layer 0', 'Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5', 'Layer 6', 'Layer 7', 'Layer 8', 'Layer 9', 'Layer 10', 'Layer 11', 'Layer 12', 'Layer 13']
  Top attended sources:      ['Layer 1(0.230)', 'Layer 2(0.165)', 'Layer 0(0.153)', 'Layer 3(0.090)', 'Layer 11(0.063)', 'Layer 10(0.048)', 'Layer 4(0.046)', 'Layer 5(0.044)']
  Constraint check: PASS

Adapter Layer 15
  Expected candidate sources: ['Embedding', 'Layer 0', 'Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5', 'Layer 6', 'Layer 7', 'Layer 8', 'Layer 9', 'Layer 10', 'Layer 11', 'Layer 12', 'Layer 13', 'Layer 14']
  Top attended sources:      ['Layer 0(0.322)', 'Embedding(0.187)', 'Layer 1(0.128)', 'Layer 2(0.116)', 'Layer 3(0.065)', 'Layer 14(0.041)', 'Layer 

## Step 7: Cross-Task Comparison with the Three Main Models

This part uses the three main task-specific models to compare how Attention Residual usage differs across tasks.


In [16]:
main_task_analyzers = {}
for task_name in ["boolq", "gsm8k", "rte"]:
    cfg = main_task_configs[task_name]
    main_task_analyzers[task_name] = analyze_task(
        task_name,
        cfg,
        main_dataloaders[task_name],
        lookback=cfg["lookback"],
        device=device,
    )

num_layers = next(iter(main_task_analyzers.values())).num_layers
combined_analyzer = AttnResAnalyzer(num_layers=num_layers, num_datasets=3, lookback=8)

for task_name, analyzer in main_task_analyzers.items():
    combined_analyzer.dataset_names[task_name] = analyzer.dataset_names.get(task_name, task_name)
    combined_analyzer.dataset_sample_counts[task_name] = len(main_datasets[task_name])

    for layer_idx in range(num_layers):
        for ds_id, alphas in analyzer.layer_source_attention[layer_idx].items():
            combined_analyzer.layer_source_attention[layer_idx][ds_id].extend(alphas)
        for ds_id, gates in analyzer.layer_gate_values[layer_idx].items():
            combined_analyzer.layer_gate_values[layer_idx][ds_id].extend(gates)
        for ds_id, norms in analyzer.layer_adapter_output_norm[layer_idx].items():
            combined_analyzer.layer_adapter_output_norm[layer_idx][ds_id].extend(norms)

stats = combined_analyzer.compute_aggregated_stats()
print("Cross-task combined analysis ready.")

print("\n" + "=" * 80)
print("TASK-SPECIFIC ATTENTION PATTERNS")
print("=" * 80)

dataset_comparison = stats["dataset_comparison"]

for dataset_id, dataset_stats in dataset_comparison.items():
    dataset_name = dataset_stats["dataset_name"]
    num_samples = dataset_stats["num_samples"]

    print(f"\n{dataset_name} ({num_samples} samples):")
    print("-" * 60)

    shown = 0
    for layer_idx in range(num_layers):
        if layer_idx in dataset_stats["layer_analysis"]:
            layer_info = dataset_stats["layer_analysis"][layer_idx]
            ranking = layer_info.get("layer_ranking", [])[:3]
            mean_gate = layer_info.get("mean_gate", None)

            print(f"  Adapter Layer {layer_idx}:")
            if ranking:
                for source_idx, weight in ranking:
                    source_name = "Embedding" if source_idx == -1 else f"Layer {source_idx}"
                    print(f"    ↖ {source_name}: {weight:.4f}")
            if mean_gate is not None:
                print(f"    Gate: {mean_gate:.6f}")
            shown += 1
        if shown >= 8:
            break


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing BoolQ ...
  processed 80 samples
✓ BoolQ analysis complete


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing GSM8K ...
  processed 80 samples
✓ GSM8K analysis complete


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing RTE ...
  processed 80 samples
✓ RTE analysis complete
Cross-task combined analysis ready.

TASK-SPECIFIC ATTENTION PATTERNS

BoolQ (100 samples):
------------------------------------------------------------
  Adapter Layer 0:
    ↖ Embedding: 1.0000
    Gate: -0.007206
  Adapter Layer 1:
    ↖ Embedding: 0.9668
    ↖ Layer 0: 0.0331
    Gate: -0.025970
  Adapter Layer 2:
    ↖ Embedding: 0.9761
    ↖ Layer 1: 0.0155
    ↖ Layer 0: 0.0086
    Gate: -0.024765
  Adapter Layer 3:
    ↖ Layer 2: 0.4895
    ↖ Embedding: 0.4883
    ↖ Layer 0: 0.0123
    Gate: -0.044281
  Adapter Layer 4:
    ↖ Embedding: 0.8262
    ↖ Layer 2: 0.0909
    ↖ Layer 3: 0.0446
    Gate: -0.034485
  Adapter Layer 5:
    ↖ Embedding: 0.9282
    ↖ Layer 3: 0.0333
    ↖ Layer 0: 0.0198
    Gate: -0.023941
  Adapter Layer 6:
    ↖ Embedding: 0.5010
    ↖ Layer 3: 0.2362
    ↖ Layer 4: 0.1509
    Gate: -0.053436
  Adapter Layer 7:
    ↖ Embedding: 0.4421
    ↖ Layer 6: 0.1646
    ↖ Layer 3: 0.1503
    Gate: -

## Step 8: Visualizations

Visualizations are generated for the task comparison analysis.


In [17]:
rte_output_dir = Path("./attnres_analysis_rte_ablation")
rte_output_dir.mkdir(parents=True, exist_ok=True)

task_output_dir = Path("./attnres_analysis_real_tasks")
task_output_dir.mkdir(parents=True, exist_ok=True)

rte_visualizer = AttnResVisualizer(output_dir=str(rte_output_dir), dpi=120)
task_visualizer = AttnResVisualizer(output_dir=str(task_output_dir), dpi=120)

print(f"RTE ablation figures will be saved to: {rte_output_dir}")
print(f"Task comparison figures will be saved to: {task_output_dir}")


RTE ablation figures will be saved to: attnres_analysis_rte_ablation
Task comparison figures will be saved to: attnres_analysis_real_tasks


In [18]:
rte_visualizer.generate_all_plots(rte_ablation_stats, rte_num_layers)
print("\n✓ RTE ablation visualizations generated!")

task_visualizer.generate_all_plots(stats, num_layers)
print("✓ Task comparison visualizations generated!")



Generating visualizations...
Output directory: attnres_analysis_rte_ablation
Saved: layer_depth_attention_heatmap.png
Saved: gate_activation_across_layers.png
Saved: attention_flow_matrix.png
Saved: layer_ranking_comparison.png
Saved: all_relation_map.png
Saved: cross_dataset_comparison.png
Saved: dataset_gate_distributions.png

All visualizations saved!

✓ RTE ablation visualizations generated!

Generating visualizations...
Output directory: attnres_analysis_real_tasks
Saved: layer_depth_attention_heatmap.png
Saved: gate_activation_across_layers.png
Saved: attention_flow_matrix.png
Saved: layer_ranking_comparison.png
Saved: all_relation_map.png
Saved: cross_dataset_comparison.png
Saved: dataset_gate_distributions.png

All visualizations saved!
✓ Task comparison visualizations generated!


## Step 9: Save Analysis Results


In [19]:
rte_ablation_analyzer.save_analysis(str(rte_output_dir / "rte_ablation_analysis.json"))
combined_analyzer.save_analysis(str(task_output_dir / "task_comparison_analysis.json"))

with open(task_output_dir / "task_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "rte_ablation_config": rte_ablation_config,
        "main_task_configs": main_task_configs,
        "max_samples_per_task": max_samples_per_task,
        "batch_size": batch_size,
    }, f, indent=2)

print(f"\n✓ RTE ablation analysis saved to {rte_output_dir}")
print(f"✓ Task comparison analysis saved to {task_output_dir}")


Analysis saved to attnres_analysis_rte_ablation/rte_ablation_analysis.json
Analysis saved to attnres_analysis_real_tasks/task_comparison_analysis.json

✓ RTE ablation analysis saved to attnres_analysis_rte_ablation
✓ Task comparison analysis saved to attnres_analysis_real_tasks


## Step 10: Key Insights Summary


In [20]:
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

print("\nPart A. RTE ablation model")
print("-" * 80)
layer_stats = rte_ablation_stats["layer_depth_stats"]
source_frequency = {}
for layer_idx, layer_stat in layer_stats.items():
    if layer_stat.get("aggregated"):
        ranking = layer_stat["aggregated"].get("layer_ranking", [])
        if ranking:
            top_source, _ = ranking[0]
            source_frequency[top_source] = source_frequency.get(top_source, 0) + 1

for source_idx, freq in sorted(source_frequency.items(), key=lambda x: x[1], reverse=True)[:10]:
    source_name = "Embedding" if source_idx == -1 else f"Layer {source_idx}"
    print(f"  {source_name}: top source for {freq} adapter layers")

print("\nPart B. Task comparison")
print("-" * 80)
for dataset_id, dataset_stats in dataset_comparison.items():
    dataset_name = dataset_stats["dataset_name"]
    first_layer = None
    for layer_idx in range(num_layers):
        if layer_idx in dataset_stats["layer_analysis"]:
            first_layer = layer_idx
            break
    if first_layer is not None:
        ranking = dataset_stats["layer_analysis"][first_layer].get("layer_ranking", [])
        if ranking:
            src, weight = ranking[0]
            src_name = "Embedding" if src == -1 else f"Layer {src}"
            print(f"  {dataset_name}: earliest analyzed layer mainly attends to {src_name} ({weight:.4f})")

print("\nGate activation snapshot")
gate_stats = stats["gate_stats"]
for layer_idx in range(min(8, num_layers)):
    gate_stat = gate_stats[layer_idx]
    if gate_stat.get("aggregated"):
        mean_gate = gate_stat["aggregated"].get("mean", 0)
        print(f"  Layer {layer_idx}: {mean_gate:.6f}")



KEY INSIGHTS

Part A. RTE ablation model
--------------------------------------------------------------------------------
  Embedding: top source for 16 adapter layers
  Layer 0: top source for 7 adapter layers
  Layer 1: top source for 1 adapter layers
  Layer 20: top source for 1 adapter layers
  Layer 3: top source for 1 adapter layers
  Layer 25: top source for 1 adapter layers
  Layer 7: top source for 1 adapter layers

Part B. Task comparison
--------------------------------------------------------------------------------
  BoolQ: earliest analyzed layer mainly attends to Embedding (1.0000)
  GSM8K: earliest analyzed layer mainly attends to Embedding (1.0000)
  RTE: earliest analyzed layer mainly attends to Embedding (1.0000)

Gate activation snapshot
  Layer 0: -0.008099
  Layer 1: -0.025701
  Layer 2: -0.032099
  Layer 3: -0.032405
  Layer 4: -0.036499
  Layer 5: 0.017232
  Layer 6: -0.052317
  Layer 7: -0.040716
